In [18]:
"""Experiments notebook

Purpose:
- Document prompt iterations and tool-call traces.
- Provide a tiny eval harness for comparing outputs across providers/models.

Notes:
- This repo’s production code lives under `src/`.
- Tooling uses mock tools by default; provider selection is config-driven.
"""

from dotenv import load_dotenv
from pathlib import Path
import json

load_dotenv()

# Optional: point to sample prompts in this notebook.
DATA_DIR = Path("./notebook/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [19]:
from src.agent.planning import build_day_by_day_plan, format_plan_markdown
from src.api.models import TripPreferences

# (Optional) If your Anthropic key has model access, you can test the live API by running the FastAPI server
# and using curl/Postman/Streamlit. This notebook stays offline/deterministic by default.


In [20]:
prefs = TripPreferences(
    origin="Amsterdam",
    destination="Copenhagen",
    daily_km=100,
    month="June",
    lodging_preference="camping",
    hostel_every_n_nights=4,
    nationality="Canadian",
    stay_days=10,
)

plan = build_day_by_day_plan(prefs)
md = format_plan_markdown(plan, preferences=prefs)
md[:800]


In [21]:
# Save one deterministic run (useful for eval snapshots)
out_path = DATA_DIR / "deterministic_example.json"
out_path.write_text(
    json.dumps(
        {
            "preferences": prefs.model_dump(),
            "markdown": md,
            "days": [d.model_dump() for d in plan],
        },
        indent=2,
    ),
    encoding="utf-8",
)
str(out_path)


AIMessage(content='<think>\n\n</think>\n\nHello! How can I assist you today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 4, 'total_tokens': 20, 'completion_time': 0.081593238, 'prompt_time': 7.001e-05, 'queue_time': 0.053644939999999995, 'total_time': 0.081663248}, 'model_name': 'deepseek-r1-distill-llama-70b', 'system_fingerprint': 'fp_76307ac09b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--c0d21a12-a07c-48f9-9891-f1b94fcee653-0', usage_metadata={'input_tokens': 4, 'output_tokens': 16, 'total_tokens': 20})

In [22]:
# Tiny eval harness (offline)
# Add prompts you care about; compare deterministic outputs across changes.

def run_eval(cases: list[TripPreferences]) -> list[dict]:
    rows = []
    for c in cases:
        p = build_day_by_day_plan(c)
        rows.append(
            {
                "origin": c.origin,
                "destination": c.destination,
                "days": len(p),
                "total_km": sum(d.distance_km for d in p),
                "has_budget": "Budget (mock)" in format_plan_markdown(p, preferences=c),
            }
        )
    return rows

cases = [
    prefs,
    prefs.model_copy(update={"daily_km": 80}),
    prefs.model_copy(update={"lodging_preference": "mixed"}),
]

run_eval(cases)


In [23]:
# Prompt iteration notes
# - If you change `src/agent/prompts/system.md`, capture before/after behaviors here.
# - You can copy/paste full conversations from the UI and store them under `notebook/data/`.


In [24]:
# End of notebook


StructuredTool(name='multiply', description='Multiply two integers.\n\nArgs:\n    a (int): The first integer.\n    b (int): The second integer.\n\nReturns:\n    int: The product of a and b.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x0000016E2181DDA0>)

NameError: name 'BaseModel' is not defined

NameError: name 'WeatherInput' is not defined